# Session 4 — Collective Communication
## From Tensor Semantics to Manual Gradient Synchronization

**How to Build Distributed AI Systems**

في الـ Session اللي فات فهمنا إن NCCL هي الـ communication engine اللي بتجهّز الـ communicator، تفهم الـ topology، تختار transport، وتقسم الـ work إلى chunks/channels.

في الـ Session دي هنغيّر السؤال من:

> **How can GPUs communicate?**

إلى:

> **What communication pattern do we actually need?**

وهنشوف الـ Collective Operations عمليًا باستخدام **PyTorch Distributed + NCCL + 2 GPUs**.

---

### Lab environment

الـ Notebook دي معمولة عشان تتشغل على بيئة فيها:

- Linux
- PyTorch
- CUDA
- **2 NVIDIA GPUs**
- NCCL backend
- `torchrun`

مثال مناسب جدًا: **Kaggle Notebook with 2× T4 GPUs**.

> **Instructor note:** الهدف هنا مش إننا نعمل Model كبير. هنستخدم Tensors صغيرة جدًا عشان الطالب يشوف الـ communication semantics بعينه.

## Learning Outcomes

بنهاية الـ Session، المفروض الطالب يقدر:

1. يشرح الفرق بين `Broadcast`, `Reduce`, `AllReduce`, `AllGather`, `ReduceScatter`, و `AllToAll`.
2. يتوقع شكل الـ tensors على كل Rank **قبل ما يشغّل الكود**.
3. يستخدم `torch.distributed` و `torchrun` لتشغيل collectives على أكتر من GPU.
4. يفهم **Collective Contract** وليه mismatch بين الـ ranks ممكن يعمل hang.
5. يفهم دور `dist.barrier()` كـ synchronization primitive.
6. يربط `AllReduce` بالـ gradient synchronization في distributed training.
7. يفهم semantic decomposition:

```text
AllReduce = ReduceScatter + AllGather
```

8. يعمل **manual gradient synchronization** بنفسه قبل ما يدخل على DDP.
9. يفرق بين:

```text
WHAT communication result do I want?
```

و:

```text
HOW NCCL implements that communication?
```

## Session Map

```text
Environment Check
      ↓
Distributed Execution Recap
      ↓
Collective Contract
      ↓
Barrier
      ↓
Broadcast
      ↓
Reduce
      ↓
AllReduce
      ↓
AllGather
      ↓
ReduceScatter
      ↓
AllToAll
      ↓
Ring AllReduce Mental Model
      ↓
Manual Gradient Synchronization
      ↓
Bridge to DDP
```

### Scope

هنركز على الـ core collectives. الحاجات دي هنذكرها كـ notes فقط ومش هندخل فيها deep dive دلوقتي:

- `Gather` / `Scatter`
- `async_op=True`
- Custom process groups
- Communication benchmarking
- Advanced overlap with CUDA streams

## 1. Environment Check

أول حاجة نتأكد إن PyTorch شايف CUDA وإن عندنا 2 GPUs.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version used by PyTorch:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {props.name} | "
        f"VRAM={props.total_memory / 1024**3:.2f} GB"
    )

In [ ]:
# Optional hardware check
!nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader

> **Expected:** `GPU count` لازم تكون 2 في الـ lab دي.

لو عندك GPU واحدة فقط، تقدر تقرأ الـ Notebook وتفهم الـ semantics، لكن الـ NCCL demos تحت محتاجة أكتر من process/GPU.

## 2. Quick Recap — `torchrun`, Rank, Local Rank, GPU

لما نشغّل:

```bash
torchrun --standalone --nproc_per_node=2 program.py
```

`torchrun` هيشغّل **2 independent Python processes**.

Conceptually:

```text
Process 0
RANK=0
LOCAL_RANK=0
        ↓
      GPU 0

Process 1
RANK=1
LOCAL_RANK=1
        ↓
      GPU 1

WORLD_SIZE=2
```

كل process بتعمل:

```python
torch.cuda.set_device(local_rank)
dist.init_process_group(backend="nccl")
```

وبكده الاتنين يبقوا أعضاء في نفس distributed process group.

> **Important:** الـ Rank هي software identity داخل الـ distributed group. هي **مش** GPU ID hardcoded جوه الـ GPU.

## 3. Why Are We Creating a Small Runtime Script from the Notebook?

Jupyter نفسها process واحدة، بينما `torchrun` محتاج يشغّل multiple independent Python processes.

عشان نخلي الـ Notebook interactive وفي نفس الوقت نشغّل NCCL صح، هنخلي الـ Notebook تكتب **temporary runtime helper** اسمه:

```text
_session4_runtime.py
```

وبعدين كل section هتشغّل demo واحدة:

```bash
torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo all_reduce
```

الـ helper ده جزء من الـ lab فقط. بعد ما نخلص فهم الـ Session، نقدر نبني `collectives_demo.py` النهائي بشكل أنضف لوحده.

In [ ]:
%%writefile _session4_runtime.py
import os
import time
import argparse

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist


def setup():
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend="nccl")

    device = torch.device(f"cuda:{local_rank}")
    return rank, local_rank, world_size, device


def cleanup():
    if dist.is_initialized():
        dist.destroy_process_group()


def fmt(tensor):
    return tensor.detach().cpu().tolist()


def ordered_print(rank, world_size, message):
    """Print rank outputs in deterministic rank order."""
    for r in range(world_size):
        dist.barrier()
        if rank == r:
            print(message, flush=True)
    dist.barrier()


def demo_identity(rank, local_rank, world_size, device):
    ordered_print(
        rank,
        world_size,
        f"Rank {rank} | LOCAL_RANK={local_rank} | device={device}"
    )
    if rank == 0:
        print(f"WORLD_SIZE={world_size}", flush=True)


def demo_barrier(rank, world_size):
    dist.barrier()
    start = time.perf_counter()

    if rank == 1:
        time.sleep(2)

    elapsed = time.perf_counter() - start
    print(f"Rank {rank} reached barrier at ~{elapsed:.2f}s", flush=True)

    dist.barrier()

    elapsed = time.perf_counter() - start
    print(f"Rank {rank} passed barrier at  ~{elapsed:.2f}s", flush=True)


def demo_broadcast(rank, world_size, device):
    tensor = (
        torch.tensor([10.0, 20.0], device=device)
        if rank == 0
        else torch.tensor([0.0, 0.0], device=device)
    )

    ordered_print(rank, world_size, f"[Rank {rank}] BEFORE: {fmt(tensor)}")
    dist.broadcast(tensor, src=0)
    ordered_print(rank, world_size, f"[Rank {rank}] AFTER : {fmt(tensor)}")


def demo_reduce(rank, world_size, device):
    if world_size != 2:
        raise RuntimeError("This teaching demo expects WORLD_SIZE=2.")

    tensor = (
        torch.tensor([1.0, 2.0], device=device)
        if rank == 0
        else torch.tensor([3.0, 4.0], device=device)
    )

    ordered_print(rank, world_size, f"[Rank {rank}] BEFORE: {fmt(tensor)}")
    dist.reduce(tensor, dst=0, op=dist.ReduceOp.SUM)

    if rank == 0:
        print(f"[Rank 0] REDUCE RESULT: {fmt(tensor)}", flush=True)
    dist.barrier()


def demo_all_reduce(rank, world_size, device):
    if world_size != 2:
        raise RuntimeError("This teaching demo expects WORLD_SIZE=2.")

    tensor = (
        torch.tensor([1.0, 2.0], device=device)
        if rank == 0
        else torch.tensor([3.0, 4.0], device=device)
    )

    ordered_print(rank, world_size, f"[Rank {rank}] BEFORE: {fmt(tensor)}")
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    ordered_print(rank, world_size, f"[Rank {rank}] AFTER : {fmt(tensor)}")


def demo_all_gather(rank, world_size, device):
    tensor = torch.tensor([float((rank + 1) * 10)], device=device)
    outputs = [torch.zeros_like(tensor) for _ in range(world_size)]

    ordered_print(rank, world_size, f"[Rank {rank}] LOCAL INPUT: {fmt(tensor)}")
    dist.all_gather(outputs, tensor)

    gathered = [x.item() for x in outputs]
    ordered_print(rank, world_size, f"[Rank {rank}] GATHERED   : {gathered}")


def demo_reduce_scatter(rank, world_size, device):
    if world_size != 2:
        raise RuntimeError("This teaching demo expects WORLD_SIZE=2.")

    input_tensor = (
        torch.tensor([1.0, 2.0, 3.0, 4.0], device=device)
        if rank == 0
        else torch.tensor([10.0, 20.0, 30.0, 40.0], device=device)
    )
    output = torch.empty(2, device=device)

    ordered_print(rank, world_size, f"[Rank {rank}] INPUT : {fmt(input_tensor)}")
    dist.reduce_scatter_tensor(output, input_tensor, op=dist.ReduceOp.SUM)
    ordered_print(rank, world_size, f"[Rank {rank}] OUTPUT: {fmt(output)}")


def demo_all_to_all(rank, world_size, device):
    if world_size != 2:
        raise RuntimeError("This teaching demo expects WORLD_SIZE=2.")

    input_tensor = (
        torch.tensor([10.0, 11.0], device=device)
        if rank == 0
        else torch.tensor([20.0, 21.0], device=device)
    )
    output = torch.empty_like(input_tensor)

    ordered_print(rank, world_size, f"[Rank {rank}] INPUT : {fmt(input_tensor)}")
    dist.all_to_all_single(output, input_tensor)
    ordered_print(rank, world_size, f"[Rank {rank}] OUTPUT: {fmt(output)}")


def demo_manual_grad_sync(rank, world_size, device):
    if world_size != 2:
        raise RuntimeError("This teaching demo expects WORLD_SIZE=2.")

    torch.manual_seed(1234)
    model = nn.Linear(2, 1, bias=False).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

    if rank == 0:
        x = torch.tensor([[1.0, 0.0], [0.0, 1.0]], device=device)
        y = torch.tensor([[1.0], [2.0]], device=device)
    else:
        x = torch.tensor([[2.0, 1.0], [1.0, 3.0]], device=device)
        y = torch.tensor([[3.0], [5.0]], device=device)

    optimizer.zero_grad()
    pred = model(x)
    loss = F.mse_loss(pred, y)
    loss.backward()

    grad = model.weight.grad

    ordered_print(
        rank,
        world_size,
        f"[Rank {rank}] local loss={loss.item():.6f} | LOCAL GRAD={fmt(grad)}"
    )

    dist.all_reduce(grad, op=dist.ReduceOp.SUM)
    grad /= world_size

    ordered_print(rank, world_size, f"[Rank {rank}] SYNCED GRAD={fmt(grad)}")

    optimizer.step()
    ordered_print(rank, world_size, f"[Rank {rank}] UPDATED WEIGHT={fmt(model.weight)}")


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--demo",
        required=True,
        choices=[
            "identity",
            "barrier",
            "broadcast",
            "reduce",
            "all_reduce",
            "all_gather",
            "reduce_scatter",
            "all_to_all",
            "manual_grad_sync",
        ],
    )
    return parser.parse_args()


def main():
    args = parse_args()
    rank, local_rank, world_size, device = setup()

    try:
        if args.demo == "identity":
            demo_identity(rank, local_rank, world_size, device)
        elif args.demo == "barrier":
            demo_barrier(rank, world_size)
        elif args.demo == "broadcast":
            demo_broadcast(rank, world_size, device)
        elif args.demo == "reduce":
            demo_reduce(rank, world_size, device)
        elif args.demo == "all_reduce":
            demo_all_reduce(rank, world_size, device)
        elif args.demo == "all_gather":
            demo_all_gather(rank, world_size, device)
        elif args.demo == "reduce_scatter":
            demo_reduce_scatter(rank, world_size, device)
        elif args.demo == "all_to_all":
            demo_all_to_all(rank, world_size, device)
        elif args.demo == "manual_grad_sync":
            demo_manual_grad_sync(rank, world_size, device)
    finally:
        cleanup()


if __name__ == "__main__":
    main()

## 4. Sanity Check — Who Am I?

قبل ما ننقل Data، نتأكد إن كل process واخدة الـ identity والـ GPU الصح.

In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo identity

Expected mental model:

```text
Rank 0 → GPU 0
Rank 1 → GPU 1
WORLD_SIZE = 2
```

> **Note:** ترتيب سطور terminal في multi-process programs مش مضمون طبيعيًا. في معظم demos هنستخدم `ordered_print()` عشان نخلي الـ output مناسبة للتدريس.

# 5. The Collective Contract

دي من أهم أفكار الـ Session كلها.

الـ collective operation مش function كل Rank تدخلها براحتها. الـ participating ranks لازم تتفق على communication sequence متوافق.

Conceptually:

```text
Rank 0: all_reduce(A) ─────┐
                           ├── same collective
Rank 1: all_reduce(B) ─────┘
```

لكن لو حصل:

```text
Rank 0: all_reduce(...)
Rank 1: broadcast(...)
```

أو Rank دخلت collective وRank تانية ما وصلتش لها أصلًا، البرنامج ممكن يعمل **hang / timeout / incorrect behavior depending on the mismatch**.

### Simple rule

> **Collectives are group operations. Think about the whole group, not one rank in isolation.**

كمان الـ tensor metadata لازم تكون compatible مع الـ operation: shapes, dtypes, device placement, split sizes… حسب الـ collective المستخدمة.

### DO NOT RUN — intentionally broken example

```python
if rank == 0:
    dist.all_reduce(tensor)
else:
    dist.broadcast(tensor, src=0)
```

الهدف من المثال إننا نفهم ليه distributed debugging مختلف: كل process ممكن تكون سليمة لو بصيت عليها لوحدها، لكن **global communication schedule** نفسه غلط.

# 6. `dist.barrier()` — Synchronization Point

`barrier()` مش بتنقل Tensor زي AllReduce. هي synchronization primitive.

الفكرة:

```text
Rank 0 ────────┐
               │
               ├── BARRIER ───→ continue
               │
Rank 1 ────────────────┘
```

محدش يكمل بعد الـ barrier لحد ما كل الـ participating ranks توصل.

هنخلي Rank 1 تتأخر ~2 seconds عمدًا. Rank 0 هتوصل بدري، لكن مش هتعدي غير لما Rank 1 توصل.

In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo barrier

### What should you observe?

تقريبًا:

```text
Rank 0 reached barrier at ~0.00s
Rank 1 reached barrier at ~2.00s

Rank 0 passed barrier at ~2.00s
Rank 1 passed barrier at ~2.00s
```

الأرقام والترتيب ممكن يختلفوا شوية، لكن الفكرة ثابتة:

> **Fast rank waits for the slow rank at the barrier.**

### Note

`barrier()` مفيدة جدًا للتجارب والتنظيم والـ debugging، لكن استخدامها زيادة عن اللزوم ممكن يمنع useful overlap ويضيف synchronization overhead.

# 7. Broadcast — One Rank → Everyone

## Before running: predict the output

```text
Rank 0 → [10, 20]
Rank 1 → [ 0,  0]
```

هنعمل:

```python
dist.broadcast(tensor, src=0)
```

مين هيبقى عنده إيه بعد الـ operation؟

### Mental model

```text
          Rank 0
        [10, 20]
        /      \
       v        v
    Rank 0    Rank 1
  [10,20]    [10,20]
```

> **Important:** `src=0` هنا معناها **Rank 0**, مش "GPU 0" كـ hardware identifier.

In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo broadcast

### Outcome

```text
BEFORE
Rank 0 → [10,20]
Rank 1 → [0,0]

AFTER BROADCAST(src=0)
Rank 0 → [10,20]
Rank 1 → [10,20]
```

**Pattern:** One-to-all.

Use cases ممكن تشمل توزيع initial state أو metadata/tensors من root rank لباقي المجموعة.

# 8. Reduce — Everyone Contributes → One Rank Gets the Reduced Result

Inputs:

```text
Rank 0 → [1,2]
Rank 1 → [3,4]
```

هنستخدم SUM:

```python
dist.reduce(tensor, dst=0, op=dist.ReduceOp.SUM)
```

Element-wise:

```text
[1,2]
+
[3,4]
=
[4,6]
```

لكن النتيجة النهائية المطلوبة بتبقى عند **destination rank فقط**.

> `dst=0` معناها Rank 0.

In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo reduce

### Mental model

```text
Rank 0 [1,2] ───┐
                 ├── SUM ───→ Rank 0 [4,6]
Rank 1 [3,4] ───┘
```

**Pattern:** Many-to-one + reduction.

### Teaching note

ما تعتمدش على contents بتاعة non-destination ranks بعد `reduce()` كأنها final result. الـ semantic result اللي يهمنا موجود عند `dst`.

# 9. AllReduce — Everyone Contributes → Everyone Gets the Reduced Result

هنستخدم نفس inputs:

```text
Rank 0 → [1,2]
Rank 1 → [3,4]
```

لكن بدل ما Rank 0 بس تاخد `[4,6]`، الاتنين هياخدوا النتيجة:

```python
dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
```

In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo all_reduce

### Result

```text
Rank 0 → [4,6]
Rank 1 → [4,6]
```

### Reduce vs AllReduce

| Operation | Who contributes? | Who gets final reduced result? |
|---|---|---|
| Reduce | All participating ranks | One destination rank |
| AllReduce | All participating ranks | Every participating rank |

وده من أهم patterns في distributed training، لأن كل worker ممكن تحسب local gradients، وبعدها كل worker تحتاج synchronized gradients قبل ما تعمل نفس model update.

# 10. AllGather — Everyone Contributes a Piece → Everyone Gets All Pieces

هنا مفيش SUM.

Inputs:

```text
Rank 0 → [10]
Rank 1 → [20]
```

After `AllGather`:

```text
Rank 0 → [10,20]
Rank 1 → [10,20]
```

الفكرة إن كل Rank عندها piece، وكل Rank في الآخر تاخد collection بتاعة كل الـ pieces.

In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo all_gather

### AllGather vs AllReduce

دي نقطة ناس كتير بتتلخبط فيها:

```text
AllGather
---------
R0: [10]
R1: [20]

→ both get [10,20]
```

مفيش element-wise reduction.

أما:

```text
AllReduce SUM
-------------
R0: [10]
R1: [20]

→ both get [30]
```

### Mental model

- **AllGather:** collect pieces.
- **AllReduce:** combine values with an operation such as SUM, then give result to everyone.

# 11. ReduceScatter — Reduce First, Then Distribute Pieces of the Result

دي مهمة جدًا لأنها هتربطنا بـ Ring AllReduce بعد شوية.

Inputs:

```text
Rank 0 → [ 1,  2,  3,  4]
Rank 1 → [10, 20, 30, 40]
```

لو عملنا element-wise SUM:

```text
[11, 22, 33, 44]
```

لكن بدل ما كل Rank تاخد النتيجة كاملة:

```text
Rank 0 → [11,22]
Rank 1 → [33,44]
```

Conceptually:

```text
REDUCE
   +
SCATTER
```

In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo reduce_scatter

### Key idea

بعد `ReduceScatter`:

- الـ reduction حصل عبر الـ ranks.
- لكن كل Rank احتفظت **بجزء** من reduced result.

وده مختلف تمامًا عن AllReduce، اللي كل Rank بعدها عندها النتيجة كاملة.

# 12. A Very Important Identity

Semantically:

```text
AllReduce
   =
ReduceScatter
   +
AllGather
```

ليه؟

### Step 1 — ReduceScatter

كل الـ ranks تساهم، نعمل reduction، وبعدها كل Rank تمسك reduced chunk مختلف.

```text
R0 owns reduced C0
R1 owns reduced C1
R2 owns reduced C2
R3 owns reduced C3
```

### Step 2 — AllGather

كل Rank تبعت reduced chunk بتاعتها للباقي.

في الآخر:

```text
Every rank owns:
[C0, C1, C2, C3]
```

وده هو AllReduce result.

> **Precision note:** دي semantic decomposition قوية جدًا لفهم Ring AllReduce، لكن معناهاش إن NCCL لازم دائمًا تنفذ كل AllReduce حرفيًا بنفس algorithm. NCCL ممكن تختار algorithms مختلفة حسب topology, message size, hardware, وغيرهم.

# 13. AllToAll — Every Rank Sends Different Pieces to Different Ranks

Inputs:

```text
Rank 0 → [10,11]
Rank 1 → [20,21]
```

اعتبر:

```text
Rank 0:
10 → destination Rank 0
11 → destination Rank 1

Rank 1:
20 → destination Rank 0
21 → destination Rank 1
```

After AllToAll:

```text
Rank 0 → [10,20]
Rank 1 → [11,21]
```

In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo all_to_all

### Why should we care?

`AllToAll` مهمة جدًا في patterns زي:

- Expert Parallelism
- Mixture-of-Experts (MoE)
- وبعض tensor/data redistribution patterns

مش هندخل في MoE هنا، لكن خليك فاكر:

> **AllToAll is about routing different pieces to different destinations.**

# 14. Collective Cheat Sheet

| Collective | Simple mental model | Final ownership |
|---|---|---|
| Broadcast | One → All | Everyone gets root's data |
| Reduce | All → One + reduction | One destination gets reduced result |
| AllReduce | All → All + reduction | Everyone gets reduced result |
| AllGather | Each contributes a piece | Everyone gets all pieces |
| ReduceScatter | Reduce + split result | Each rank gets one reduced piece |
| AllToAll | Everyone sends different pieces everywhere | Each rank receives its destined pieces |
| Barrier | Synchronize only | No tensor result |

### Quick test

لو قلت لك:

```text
Rank 0 → [1]
Rank 1 → [2]
```

- AllReduce SUM → ?
- AllGather → ?
- Reduce SUM to Rank 0 → ?

الإجابة:

```text
AllReduce SUM:
R0 [3]
R1 [3]

AllGather:
R0 [1,2]
R1 [1,2]

Reduce SUM to R0:
R0 [3]
```

# 15. Ring AllReduce — Now We Know WHAT, Let's Discuss HOW

لحد هنا `dist.all_reduce()` بالنسبة لنا لها semantics واضحة:

> كل Rank تساهم، وكل Rank تستلم reduced result.

لكن إزاي ننفذ ده efficiently؟

واحدة من أشهر الخطط هي **Ring AllReduce**.

هنستخدم 4 ranks conceptually عشان الـ ring تبقى واضحة:

```text
R0 → R1 → R2 → R3 → R0
```

وكل Rank عندها Tensor:

| Rank | C0 | C1 | C2 | C3 |
|---|---:|---:|---:|---:|
| R0 | 1 | 10 | 100 | 1000 |
| R1 | 2 | 20 | 200 | 2000 |
| R2 | 3 | 30 | 300 | 3000 |
| R3 | 4 | 40 | 400 | 4000 |

هدف AllReduce SUM:

```text
[10, 100, 1000, 10000]
```

على **كل Rank**.

## 15.1 Understand One Chunk First — C0

ما تبصش على الأربع chunks مرة واحدة.

خد C0 فقط:

```text
R0 has 1
R1 has 2
R2 has 3
R3 has 4
```

رحلة partial sum:

```text
R0 sends 1 → R1
R1: 1 + 2 = 3

R1 sends 3 → R2
R2: 3 + 3 = 6

R2 sends 6 → R3
R3: 6 + 4 = 10
```

إذن reduced value لـ C0 بقت:

```text
C0 = 10
```

الفكرة المهمة: الـ partial result بيتحرك ويُدمج مع local contribution في كل خطوة.

## 15.2 Now Let Different Chunks Move at the Same Time

لو عملنا whole tensor كـ one sequential object، هنسيب links/resources كتير underutilized.

عشان كده بنقسم data إلى chunks، ونخلّي chunks مختلفة تتحرك بالتوازي.

### ReduceScatter — Round 1

```text
R0 sends C0=1    → R1 → 1+2       = 3
R1 sends C1=20   → R2 → 20+30     = 50
R2 sends C2=300  → R3 → 300+400   = 700
R3 sends C3=4000 → R0 → 4000+1000 = 5000
```

### Round 2

```text
C0: 3    → R2 → +3    = 6
C1: 50   → R3 → +40   = 90
C2: 700  → R0 → +100  = 800
C3: 5000 → R1 → +2000 = 7000
```

### Round 3

```text
C0: 6    → R3 → +4    = 10
C1: 90   → R0 → +10   = 100
C2: 800  → R1 → +200  = 1000
C3: 7000 → R2 → +3000 = 10000
```

دلوقتي reduced chunks موزعة:

```text
R3 owns C0 = 10
R0 owns C1 = 100
R1 owns C2 = 1000
R2 owns C3 = 10000
```

دي **ReduceScatter phase**.

## 15.3 AllGather Phase

دلوقتي مفيش reduction إضافية مطلوبة.

كل Rank عندها reduced chunk واحدة، والمطلوب إن الأربع chunks يوصلوا لكل ranks.

```text
R0 has C1
R1 has C2
R2 has C3
R3 has C0
```

عن طريق ring exchange، الـ chunks المكتملة تنتشر لحد ما كل Rank تبقى عندها:

```text
[C0, C1, C2, C3]
=
[10, 100, 1000, 10000]
```

إذن:

```text
Ring AllReduce
=
Ring ReduceScatter
+
Ring AllGather
```

### Why chunks matter

مش عشان الـ math محتاجة chunks.

الـ chunks بتساعد إن أكتر من Rank / link يشتغلوا في نفس الوقت ونقدر نعمل pipelining ونستخدم communication resources أحسن.

> **Important:** Ring هو algorithm / communication plan، مش transport. الـ physical path نفسه ممكن يكون PCIe / NVLink / network، والـ transport mechanism ممكن تكون P2P / SHM / NET حسب الحالة.

# 16. The Training Connection — Why Does AllReduce Matter?

دلوقتي نربط الكلام ده بالـ training.

في Data Parallel style training:

```text
Same Model
   │
   ├── Rank 0 gets Batch A
   │       ↓
   │    Forward
   │       ↓
   │    Backward
   │       ↓
   │    Local Gradients
   │
   └── Rank 1 gets Batch B
           ↓
        Forward
           ↓
        Backward
           ↓
        Local Gradients
```

بما إن الـ batches مختلفة، الـ local gradients غالبًا هتختلف.

لو كل Rank عملت `optimizer.step()` فورًا:

```text
Model on Rank 0 changes one way
Model on Rank 1 changes another way
```

بعد شوية الـ replicas هتتباعد.

الحل:

```text
Local Gradients
      ↓
AllReduce SUM
      ↓
Divide by WORLD_SIZE
      ↓
Same Averaged Gradient
      ↓
optimizer.step()
      ↓
Same Parameter Update
```

## 16.1 Manual Gradient Synchronization

هنعمل Linear model صغيرة جدًا.

مهم جدًا:

- نفس initial weights على الاتنين ranks.
- كل Rank تاخد local batch مختلفة.
- نعمل `backward()`.
- نطبع gradients **قبل synchronization**.
- نعمل `all_reduce`.
- نقسم على `world_size`.
- نطبع gradients **بعد synchronization**.
- نعمل optimizer step ونشوف إن updated weights متطابقة.

### Predict before running

قبل الـ AllReduce:

```text
Rank 0 gradient != Rank 1 gradient
```

بعد الـ AllReduce + average:

```text
Rank 0 gradient == Rank 1 gradient
```

In [ ]:
!torchrun --standalone --nproc_per_node=2 _session4_runtime.py --demo manual_grad_sync

### What just happened?

إحنا عملنا يدويًا جزء أساسي جدًا من distributed data-parallel training:

```python
loss.backward()

dist.all_reduce(param.grad, op=dist.ReduceOp.SUM)
param.grad /= world_size

optimizer.step()
```

### Important precision note

القسمة البسيطة على `world_size` هنا بتدي average صحيح للـ local mean gradients لأن الـ demo عندها **نفس local batch size على كل Rank**.

لو local batch sizes مختلفة، averaging الصحيح للـ global examples محتاج weighting مناسب، ومينفعش نفترض إن divide-by-world-size دائمًا يمثل global sample mean.

# 17. Bridge to DDP

دلوقتي لما نشوف:

```python
from torch.nn.parallel import DistributedDataParallel as DDP

model = DDP(
    model,
    device_ids=[local_rank]
)
```

المفروض ما تبقاش شايفها magic.

Conceptually:

```text
Forward
   ↓
Backward
   ↓
Gradient becomes ready
   ↓
DDP coordinates gradient synchronization
   ↓
Collective communication through ProcessGroupNCCL
   ↓
NCCL moves/reduces the data
   ↓
Synchronized gradients
```

في implementation حقيقية، DDP مش ببساطة تعمل Python `all_reduce()` واحدة لكل parameter زي demo بتاعتنا.

هي تستخدم mechanisms زي **gradient buckets** بحيث communication تقدر تبدأ أثناء backward computation بدل ما تستنى كل gradients تخلص.

وده هيبقى transition ممتاز للـ DDP internals later.

# 18. Common Mistakes & Debugging Notes

### 1. Different collective order across ranks

```text
Rank 0:
all_reduce()
broadcast()

Rank 1:
broadcast()
all_reduce()
```

مشكلة. الـ collective schedule لازم يكون compatible.

---

### 2. One rank never reaches the collective

ممكن Rank تكون:

- crashed
- OOM
- stuck in dataloader/code path مختلف
- دخلت branch مختلفة

فتلاقي باقي ranks واقفة مستنية.

---

### 3. Confusing rank with GPU index

```python
src=0
dst=0
```

دول ranks، مش physical GPU IDs.

---

### 4. Shape / split mismatches

Collectives مختلفة عندها input/output contracts مختلفة. خصوصًا `ReduceScatter` و `AllToAll` لازم تفكر في تقسيم الـ data بوضوح.

---

### 5. Too many barriers

`barrier()` مفيدة في teaching/debugging، لكن مش لازم تحطها في كل مكان في production code. الـ unnecessary synchronization ممكن يقلل parallelism.

---

### 6. Printing from every rank

Multi-process logs بتتداخل. أثناء التدريس استخدم rank-prefixed logs وbarriers للتنظيم عند الحاجة.

# 19. Things We Intentionally Did NOT Deep Dive Into

## Gather / Scatter

مفيدين، لكن الـ core mental model اتغطى كويس بالـ operations اللي فوق.

## `async_op=True`

PyTorch collectives تقدر تدعم asynchronous work handles في APIs معينة. ده مهم جدًا للـ overlap، لكن محتاج section مستقل عن synchronization وCUDA stream semantics.

## Custom Process Groups

مش كل communication لازم تشمل `WORLD_SIZE` كله. نقدر نعمل subgroups لاحقًا، وده مهم جدًا في Tensor Parallelism / Pipeline Parallelism / Expert Parallelism.

## Performance Benchmarking

لاحقًا هنقيس latency, effective bandwidth, message-size effects, `nccl-tests`, والـ overlap. لكن الأول كان لازم semantics نفسها تبقى واضحة.

## Advanced algorithms

Ring مش algorithm الوحيدة. NCCL ممكن تستخدم plans مختلفة حسب البيئة. هدفنا هنا إن Ring تبني intuition قوية لـ chunked collective communication.

# 20. Final Mental Model

```text
Application needs a communication result
              ↓
Choose a Collective
              ↓
WHAT result?
Broadcast / AllReduce / AllGather / ...
              ↓
NCCL chooses an execution strategy
              ↓
HOW to move it?
Algorithm + chunks + channels + protocol + transport
              ↓
Physical interconnect
              ↓
GPU kernels / network progress
              ↓
Bytes move
```

### The key distinction

**Collective = semantics.**

يعني:

> What result should the group produce?

أما Ring / Tree / transport / protocol / channels فهم implementation/execution decisions تحت الـ collective abstraction.

# 21. Session Outcomes Checklist

بعد ما تخلص الـ Notebook، جرّب تجاوب من غير ما تبص فوق:

- [ ] What is the difference between Reduce and AllReduce?
- [ ] What is the difference between AllGather and AllReduce?
- [ ] Why is ReduceScatter useful?
- [ ] What happens if ranks call collectives in incompatible order?
- [ ] What does `barrier()` guarantee?
- [ ] Why can local gradients differ between ranks?
- [ ] Why does gradient synchronization keep model replicas aligned?
- [ ] Why is `AllReduce = ReduceScatter + AllGather` a useful mental model?
- [ ] Why is Ring an algorithm, not a transport?
- [ ] Why doesn't `backend="nccl"` itself tell you which physical interconnect is used?

لو الإجابات واضحة، فأنت جاهز تدخل على **DDP internals** من غير ما تكون `DistributedDataParallel` بالنسبة لك black box.

# References

للمراجعة بعد الـ Session:

- **PyTorch Distributed documentation** — `torch.distributed` collectives and process groups  
  https://docs.pytorch.org/docs/stable/distributed.html

- **PyTorch DistributedDataParallel documentation** — gradient buckets and distributed training behavior  
  https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html

- **NVIDIA NCCL documentation** — collective communication primitives and NCCL concepts  
  https://docs.nvidia.com/deeplearning/nccl/

> الـ Notebook دي intentionally teaching-oriented: الأمثلة صغيرة وواضحة عشان الـ communication semantics تبقى مرئية قبل ما ندخل في performance and framework internals.